# Práctica 4 — Fine-tuning con LoRA/QLoRA (Nivel 10)
**Tarea:** ajustar un modelo instruct pequeño con el dataset de la Práctica 3 (*redactor de respuestas a preguntas frecuentes de RRHH*) y obtener un adaptador funcional.

**Cómo usar este cuaderno en Google Colab**
1. `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`.
2. Ejecute las celdas **en orden** (`Entorno de ejecución → Ejecutar todo` también sirve, salvo la exportación opcional a GGUF, que está desactivada por defecto).
3. Cuando se le pida, suba `train.jsonl`, `val.jsonl` y `test.jsonl` (los entregables de la P3).
4. Al final, descargue `entregables_p4.zip`.

| Paso de la práctica | Sección | Rúbrica |
|---|---|---|
| 1 Entorno y modelo cuantizado | 1, 3, 4 | Configuración (5) |
| 2 Línea base (10 ejemplos de test) | 6 | — |
| 3 Configurar LoRA (r=16, α=32, dropout 0.05, lr 2e-4, 2 épocas) | 2, 7 | Configuración (5) |
| 4 Entrenar y registrar pérdidas | 7 | Entrenamiento (5) |
| 5 Segunda configuración y sobreajuste | 8 | Experimento (5) |
| 6 Probar el adaptador | 7–9 | Análisis (5) |
| 7 Exportar (adaptador, fusión, GGUF, Ollama) | 11, 12 | — |

> Los resultados numéricos (pérdidas, tiempos, respuestas) los produce **su** ejecución; pueden variar levemente entre GPU y versiones de las bibliotecas.

## 1. Entorno

In [ ]:
# Instalación (≈1-2 min). `trl` no se usa aquí: el enmascarado de la pérdida se hace de forma explícita (ver sección 5),
# lo que evita depender de cambios de API entre versiones de TRL.
!pip install -q -U transformers peft datasets bitsandbytes accelerate matplotlib pandas

In [ ]:
import json, os, sys, time, gc, math, re, pathlib, platform
import torch, transformers, peft, datasets, accelerate
try:
    import bitsandbytes
except Exception as e:
    raise SystemExit("No se pudo importar bitsandbytes. Use 'Entorno de ejecución → Reiniciar sesión' y ejecute desde esta celda. Detalle: %s" % e)

assert torch.cuda.is_available(), "No hay GPU: menú 'Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)'."
gpu = torch.cuda.get_device_properties(0)
BF16 = torch.cuda.get_device_capability(0)[0] >= 8   # solo Ampere o superior. OJO: is_bf16_supported() devuelve True en una T4 (emulado, lento y rompe fp16)
COMPUTO = torch.bfloat16 if BF16 else torch.float16

os.makedirs("resultados", exist_ok=True)
entorno = (f"python {platform.python_version()}\ntorch {torch.__version__}\ntransformers {transformers.__version__}\n"
           f"peft {peft.__version__}\ndatasets {datasets.__version__}\naccelerate {accelerate.__version__}\n"
           f"bitsandbytes {bitsandbytes.__version__}\nGPU {gpu.name} ({gpu.total_memory/1e9:.1f} GB) | bf16={BF16}\n")
open("resultados/entorno.txt", "w").write(entorno)
print(entorno)

## 2. Parámetros del experimento
- **Modelo base:** `unsloth/Llama-3.2-3B-Instruct` (copia abierta de Llama 3.2 3B Instruct, sin solicitud de acceso). Elija según su VRAM:

| Modelo | VRAM aprox. con QLoRA | Cuándo |
|---|---|---|
| `unsloth/Llama-3.2-1B-Instruct` | ≥ 6 GB | GPU modesta o si hay errores de memoria |
| `unsloth/Llama-3.2-3B-Instruct` | ≥ 8 GB | **Recomendado en Colab T4** |
| `unsloth/Meta-Llama-3.1-8B-Instruct` | ≥ 16 GB (justo en T4) | Mejor calidad, más lento |

- **Configuración A** = la que pide la práctica. **Configuración B** = igual que A pero con **4 épocas**: cambiar una sola variable aísla el efecto de la duración y permite ver si aparece sobreajuste. (Otras opciones válidas: `r=8` con 1 época, o 3 épocas.)

In [ ]:
MODELO = "unsloth/Llama-3.2-3B-Instruct"
MAX_LEN = 512             # tokens máximos por ejemplo (se verifica en la sección 5)
SEMILLA = 42
N_EVAL = 10               # ejemplos del conjunto de prueba para línea base y comparación
MAX_NUEVOS_TOKENS = 200   # longitud máxima de las respuestas generadas
BATCH, ACUM = 4, 2        # lote efectivo = 8. Si hay error de memoria: BATCH=2, ACUM=4
EVAL_STEPS = 3            # cada cuántos pasos se mide la pérdida de validación
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

CONFIGS = {
    "A": dict(r=16, alpha=32, dropout=0.05, lr=2e-4, epocas=2),   # obligatoria (paso 3)
    "B": dict(r=16, alpha=32, dropout=0.05, lr=2e-4, epocas=4),   # experimento (paso 5)
    # "C": dict(r=8, alpha=16, dropout=0.05, lr=2e-4, epocas=1),  # alternativa del enunciado
}
for k, v in CONFIGS.items():
    print(k, v)

## 3. Datos de la Práctica 3
Suba `train.jsonl`, `val.jsonl` y `test.jsonl` (formato chat: `system` → `user` → `assistant`).

In [ ]:
from datasets import Dataset
DATA_DIR = pathlib.Path("/content/datos"); DATA_DIR.mkdir(exist_ok=True)

faltan = [n for n in ("train", "val", "test") if not (DATA_DIR / f"{n}.jsonl").exists()]
if faltan:
    from google.colab import files
    print("Seleccione los tres archivos: train.jsonl, val.jsonl y test.jsonl")
    for nombre, contenido in files.upload().items():
        nombre_limpio = re.sub(r" \(\d+\)", "", nombre)      # "train (1).jsonl" -> "train.jsonl"
        (DATA_DIR / nombre_limpio).write_bytes(contenido)
# Alternativa con Google Drive:
#   from google.colab import drive; drive.mount('/content/drive'); DATA_DIR = pathlib.Path('/content/drive/MyDrive/p3')

def cargar(nombre):
    ruta = DATA_DIR / f"{nombre}.jsonl"
    filas = [json.loads(l) for l in ruta.read_text(encoding="utf-8").splitlines() if l.strip()]
    for i, f in enumerate(filas, 1):
        roles = [m["role"] for m in f["messages"]]
        assert roles == ["system", "user", "assistant"], f"{nombre}.jsonl, línea {i}: roles {roles}"
    return [f["messages"] for f in filas]

train, val, test = cargar("train"), cargar("val"), cargar("test")
SISTEMA = train[0][0]["content"]
print(f"train={len(train)}  val={len(val)}  test={len(test)}")
assert len(test) >= N_EVAL, f"test.jsonl tiene {len(test)} ejemplos; se necesitan al menos {N_EVAL}"
print("\nEjemplo:", train[0][1]["content"], "\n ->", train[0][2]["content"])

## 4. Modelo base cuantizado (4 bits, NF4) y tokenizador
QLoRA mantiene los pesos del modelo base congelados en 4 bits y entrena solo los adaptadores LoRA (en 16 bits).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments, set_seed
from packaging.version import Version

tokenizer = AutoTokenizer.from_pretrained(MODELO)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
PAD_ID = tokenizer.pad_token_id

def _kw_dtype(dtype):            # 'torch_dtype' se renombró a 'dtype' en transformers 4.56
    return {"dtype": dtype} if Version(transformers.__version__) >= Version("4.56") else {"torch_dtype": dtype}

def cargar_base():
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=COMPUTO)
    return AutoModelForCausalLM.from_pretrained(MODELO, quantization_config=bnb,
                                                device_map={"": 0}, **_kw_dtype(COMPUTO))

def liberar():
    """Llámela DESPUÉS de poner a None la variable del modelo (p. ej. `modelo = None; liberar()`)."""
    gc.collect(); torch.cuda.empty_cache()

print("Tokenizador listo. Pad token:", tokenizer.pad_token)

## 5. Preparación de los ejemplos: la pérdida se calcula **solo sobre la respuesta**
Cada ejemplo se convierte con la plantilla de chat del modelo. Los tokens del *prompt* (system + user) llevan la etiqueta `-100` (la pérdida los ignora) y solo se entrena la respuesta del asistente, incluido el token de fin de turno para que el modelo aprenda a detenerse.

In [ ]:
def a_ids(mensajes, generation_prompt):
    texto = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=generation_prompt)
    return tokenizer(texto, add_special_tokens=False)["input_ids"]      # la plantilla ya incluye el BOS

def tokenizar(mensajes):
    prompt = a_ids(mensajes[:-1], True)
    completo = a_ids(mensajes, False)
    if completo[:len(prompt)] != prompt:
        raise ValueError("El prompt no es prefijo del ejemplo completo: revise la plantilla de chat del modelo.")
    return {"input_ids": completo, "attention_mask": [1] * len(completo),
            "labels": [-100] * len(prompt) + completo[len(prompt):]}

def rellenar(lote, pad_id):
    n = max(len(x["input_ids"]) for x in lote)
    return {
        "input_ids":      [x["input_ids"] + [pad_id] * (n - len(x["input_ids"])) for x in lote],
        "attention_mask": [x["attention_mask"] + [0] * (n - len(x["attention_mask"])) for x in lote],
        "labels":         [x["labels"] + [-100] * (n - len(x["labels"])) for x in lote],
    }

class Colador:
    def __init__(self, pad_id): self.pad_id = pad_id
    def __call__(self, lote):
        return {k: torch.tensor(v) for k, v in rellenar(lote, self.pad_id).items()}

tok_train = [tokenizar(m) for m in train]
tok_val = [tokenizar(m) for m in val]
longitudes = [len(x["input_ids"]) for x in tok_train + tok_val]
print(f"Tokens por ejemplo: media {sum(longitudes)/len(longitudes):.0f}, máx. {max(longitudes)} (límite {MAX_LEN})")
assert max(longitudes) <= MAX_LEN, "Hay ejemplos más largos que MAX_LEN: súbalo o acorte los ejemplos."

ds_train, ds_val = Dataset.from_list(tok_train), Dataset.from_list(tok_val)
ejemplo = tok_train[0]
print("\n--- Prompt (NO entra en la pérdida) ---")
print(tokenizer.decode([t for t, l in zip(ejemplo["input_ids"], ejemplo["labels"]) if l == -100]))
print("--- Respuesta (SÍ entra en la pérdida) ---")
print(tokenizer.decode([l for l in ejemplo["labels"] if l != -100]))

## 6. Línea base: respuestas del modelo **sin** entrenar (paso 2)
Generación determinista (`do_sample=False`) para que la comparación sea reproducible. Se usan los primeros `N_EVAL` ejemplos de `test.jsonl`.

In [ ]:
@torch.no_grad()
def generar(modelo, mensajes, max_nuevos=MAX_NUEVOS_TOKENS):
    texto = tokenizer.apply_chat_template(mensajes[:-1], tokenize=False, add_generation_prompt=True)
    entrada = tokenizer(texto, add_special_tokens=False, return_tensors="pt").to(modelo.device)
    modelo.eval(); modelo.config.use_cache = True
    salida = modelo.generate(**entrada, max_new_tokens=max_nuevos, do_sample=False, temperature=None, top_p=None,
                             pad_token_id=PAD_ID)
    return tokenizer.decode(salida[0, entrada["input_ids"].shape[1]:], skip_special_tokens=True).strip()

RESPUESTAS = {}
def probar(modelo, nombre):
    res = [{"pregunta": m[1]["content"], "referencia": m[2]["content"], "respuesta": generar(modelo, m)}
           for m in test[:N_EVAL]]
    RESPUESTAS[nombre] = res
    json.dump(res, open(f"resultados/respuestas_{nombre}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    return res

t0 = time.time()
base = cargar_base()
probar(base, "base")
print(f"Línea base generada en {time.time()-t0:.0f} s\n")
for r in RESPUESTAS["base"][:2]:
    print("P:", r["pregunta"], "\nREF:", r["referencia"], "\nBASE:", r["respuesta"], "\n")
base = None; liberar()

## 7. Configuración A (paso 3–4): LoRA r=16, α=32, dropout 0.05, lr 2e-4, 2 épocas
La función `entrenar` carga el modelo cuantizado, añade los adaptadores LoRA, mide la pérdida de validación **antes** de entrenar (paso 0) y cada `EVAL_STEPS` pasos, y guarda el adaptador y el registro completo.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

LOGS = {}

def entrenar(nombre, cfg):
    set_seed(SEMILLA); torch.cuda.reset_peak_memory_stats()
    modelo = cargar_base()
    modelo = prepare_model_for_kbit_training(modelo)
    modelo = get_peft_model(modelo, LoraConfig(r=cfg["r"], lora_alpha=cfg["alpha"], lora_dropout=cfg["dropout"],
                                               bias="none", task_type="CAUSAL_LM", target_modules=TARGET_MODULES))
    for p in modelo.parameters():                       # el GradScaler fp16 exige adaptadores en float32
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.to(torch.float32)
    modelo.config.use_cache = False
    entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    total = sum(p.numel() for p in modelo.parameters())
    print(f"[{nombre}] parámetros entrenables: {entrenables:,} ({100*entrenables/total:.2f}% del total)")

    args = TrainingArguments(
        output_dir=f"/content/salida_{nombre}", per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
        gradient_accumulation_steps=ACUM, num_train_epochs=cfg["epocas"], learning_rate=cfg["lr"],
        lr_scheduler_type="cosine", warmup_steps=2, weight_decay=0.0, optim="paged_adamw_8bit",
        fp16=not BF16, bf16=BF16, gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=1, eval_strategy="steps", eval_steps=EVAL_STEPS, save_strategy="no",
        seed=SEMILLA, report_to="none", remove_unused_columns=False)
    trainer = Trainer(model=modelo, args=args, train_dataset=ds_train, eval_dataset=ds_val, data_collator=Colador(PAD_ID))

    trainer.evaluate()                                  # pérdida de validación antes de entrenar (paso 0)
    t0 = time.time(); trainer.train(); seg = time.time() - t0
    trainer.evaluate()                                  # evaluación final

    LOGS[nombre] = {"cfg": cfg, "log": trainer.state.log_history, "segundos": round(seg, 1),
                    "vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
                    "entrenables": entrenables, "total": total,
                    "pasos": trainer.state.global_step}
    json.dump(LOGS[nombre], open(f"resultados/log_{nombre}.json", "w"), indent=2)
    ruta = f"adaptadores/adaptador_{nombre}"
    modelo.save_pretrained(ruta); tokenizer.save_pretrained(ruta)
    print(f"[{nombre}] listo en {seg:.0f} s | VRAM máx. {LOGS[nombre]['vram_gb']} GB | adaptador en {ruta}")
    return modelo

modelo_A = entrenar("A", CONFIGS["A"])

### Probar el adaptador A (paso 6) con los mismos ejemplos de la línea base

In [ ]:
probar(modelo_A, "A")
for r in RESPUESTAS["A"][:2]:
    print("P:", r["pregunta"], "\nREF:", r["referencia"], "\nA:", r["respuesta"], "\n")
modelo_A = None; liberar()

## 8. Configuración B (paso 5): mismos hiperparámetros, 4 épocas

In [ ]:
modelo_B = entrenar("B", CONFIGS["B"])
probar(modelo_B, "B")
for r in RESPUESTAS["B"][:2]:
    print("P:", r["pregunta"], "\nREF:", r["referencia"], "\nB:", r["respuesta"], "\n")
modelo_B = None; liberar()

## 9. Curvas de pérdida y diagnóstico de sobreajuste
Eje X en **épocas** para poder comparar A y B. La pérdida de validación en el paso 0 es la del modelo sin entrenar. Regla orientativa: hay indicio de sobreajuste si la pérdida de validación final supera en más de un 5 % su mínimo mientras la de entrenamiento sigue bajando. Con solo 12 ejemplos de validación la medida es ruidosa: úsela junto con las respuestas generadas.

In [ ]:
import matplotlib.pyplot as plt

def serie(log, clave):
    return [(e.get("epoch", 0.0), e["step"], e[clave]) for e in log if clave in e]

def diagnostico(log):
    ev = serie(log, "eval_loss"); tr = serie(log, "loss")
    inicial = ev[0][2]
    entrenado = [x for x in ev if x[1] > 0]
    mejor = min(entrenado, key=lambda x: x[2]); final = entrenado[-1]
    sube = (final[2] - mejor[2]) / mejor[2]
    train_final = sum(x[2] for x in tr[-3:]) / len(tr[-3:])
    return {"val_inicial": inicial, "val_min": mejor[2], "epoca_min": mejor[0], "paso_min": mejor[1],
            "val_final": final[2], "train_final": train_final, "brecha": final[2] - train_final,
            "sube_pct": 100 * sube,
            "sobreajuste": sube > 0.05 and tr[-1][2] < tr[len(tr) // 2][2]}

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
colores = {"A": "tab:blue", "B": "tab:orange", "C": "tab:green"}
for n, d in LOGS.items():
    tr, ev = serie(d["log"], "loss"), serie(d["log"], "eval_loss")
    ax[0].plot([x[0] for x in tr], [x[2] for x in tr], color=colores.get(n), label=f"{n}: {d['cfg']['epocas']} épocas")
    ax[1].plot([x[0] for x in ev], [x[2] for x in ev], "o-", color=colores.get(n), label=f"{n}: {d['cfg']['epocas']} épocas")
ax[0].set_title("Pérdida de entrenamiento"); ax[1].set_title("Pérdida de validación")
for a in ax:
    a.set_xlabel("época"); a.set_ylabel("pérdida (cross-entropy)"); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.savefig("resultados/curvas_perdida.png", dpi=150); plt.show()

DIAG = {n: diagnostico(d["log"]) for n, d in LOGS.items()}
for n, d in DIAG.items():
    print(f"[{n}] val inicial {d['val_inicial']:.3f} → mínima {d['val_min']:.3f} (época {d['epoca_min']:.2f}, paso {d['paso_min']}) "
          f"→ final {d['val_final']:.3f} | train final {d['train_final']:.3f} | brecha {d['brecha']:+.3f} | "
          f"final vs mínima {d['sube_pct']:+.1f}% → {'POSIBLE SOBREAJUSTE' if d['sobreajuste'] else 'sin evidencia clara de sobreajuste'}")

## 10. Comparación de respuestas: base vs. A vs. B
Métricas objetivas de **formato y estilo** (la guía de estilo de la P3), calculadas sobre los 10 ejemplos de prueba. Miden si el modelo adoptó el estilo, **no** si el dato es correcto: eso requiere la revisión manual de la tabla `comparacion.csv`.

In [ ]:
import pandas as pd

def tokens(t): return re.findall(r"\w+", t.lower())
def f1(a, b):
    ta, tb = tokens(a), tokens(b)
    comunes = sum(min(ta.count(w), tb.count(w)) for w in set(ta))
    if not ta or not tb or comunes == 0: return 0.0
    p, r = comunes / len(ta), comunes / len(tb)
    return 2 * p * r / (p + r)
def ultima(t): return t.strip().split("\n")[-1]
def tipo_cierre(t):
    u = ultima(t)
    return "fuente" if u.startswith("Fuente: Reglamento de RRHH, secci") else ("derivacion" if u.startswith("Derivación:") else "ninguno")
def secciones(t):
    u = ultima(t)
    return set(re.findall(r"\d+(?:\.\d+)?", u.split("secci", 1)[1])) if "secci" in u and u.startswith("Fuente") else set()
def plano(t): return not re.search(r"\*\*|`|^#{1,6}\s|^\s*[-*•]\s|[\U0001F300-\U0001FAFF]", t, re.M)

filas = []
for modelo, res in RESPUESTAS.items():
    for i, r in enumerate(res):
        filas.append({"ejemplo": i + 1, "modelo": modelo, "pregunta": r["pregunta"], "referencia": r["referencia"],
                      "respuesta": r["respuesta"],
                      "cierre_correcto": tipo_cierre(r["respuesta"]) != "ninguno",
                      "mismo_tipo_cierre": tipo_cierre(r["respuesta"]) == tipo_cierre(r["referencia"]),
                      "misma_seccion": secciones(r["respuesta"]) == secciones(r["referencia"]),
                      "max_100_palabras": len(r["respuesta"].split()) <= 100,
                      "texto_plano": plano(r["respuesta"]),
                      "f1_vs_referencia": round(f1(r["respuesta"], r["referencia"]), 3)})
df = pd.DataFrame(filas)
df["revision_manual_1a5"] = ""            # complete a mano: 1 = incorrecta … 5 = correcta y bien redactada
df.to_csv("resultados/comparacion.csv", index=False, encoding="utf-8-sig")
RESUMEN = df.groupby("modelo")[["cierre_correcto", "mismo_tipo_cierre", "misma_seccion", "max_100_palabras",
                                "texto_plano", "f1_vs_referencia"]].mean().round(2).reindex(list(RESPUESTAS))
RESUMEN

In [ ]:
# Lado a lado (los 3 primeros ejemplos)
for i in range(3):
    print("=" * 90, f"\nPREGUNTA {i+1}: {RESPUESTAS['base'][i]['pregunta']}\n\nREFERENCIA:\n{RESPUESTAS['base'][i]['referencia']}")
    for n in RESPUESTAS:
        print(f"\n[{n}]\n{RESPUESTAS[n][i]['respuesta']}")

## 11. Borrador del análisis de una página
La celda genera `analisis_borrador.md` con **sus** cifras ya insertadas. Complete las secciones marcadas con ✎ con su interpretación; es lo que se evalúa en "análisis e interpretación".

In [ ]:
def fila(n):
    d, c, g = LOGS[n], LOGS[n]["cfg"], DIAG[n]
    return (f"| {n} | {c['r']} / {c['alpha']} / {c['dropout']} | {c['lr']} | {c['epocas']} | {d['pasos']} | "
            f"{d['entrenables']/1e6:.1f} M | {d['segundos']/60:.1f} min | {d['vram_gb']} GB |")

tabla_cfg = "\n".join(fila(n) for n in LOGS)
tabla_val = "\n".join(
    f"| {n} | {g['val_inicial']:.3f} | {g['val_min']:.3f} (época {g['epoca_min']:.2f}) | {g['val_final']:.3f} | {g['train_final']:.3f} | "
    f"{g['brecha']:+.3f} | {g['sube_pct']:+.1f} % | {'sí' if g['sobreajuste'] else 'no'} |" for n, g in DIAG.items())
tabla_fmt = "| modelo | cierre | mismo tipo cierre | misma sección | ≤100 palabras | texto plano | F1 |\n|---|---|---|---|---|---|---|\n" + "\n".join(
    f"| {m} | " + " | ".join(f"{v:.0%}" if k != "f1_vs_referencia" else f"{v:.2f}" for k, v in fila_.items()) + " |"
    for m, fila_ in RESUMEN.iterrows())

borrador = f"""# Análisis — Práctica 4: LoRA sobre `{MODELO}`
Dataset: P3 (RRHH FAQ) · {len(train)} train / {len(val)} val / {len(test)} test · evaluación sobre {N_EVAL} ejemplos de test · GPU: {gpu.name}

## 1. Configuraciones
| Config | r / α / dropout | lr | épocas | pasos | parám. entrenables | tiempo | VRAM pico |
|---|---|---|---|---|---|---|---|
{tabla_cfg}

## 2. Curvas de pérdida (`curvas_perdida.png`)
| Config | val inicial | val mínima | val final | train final | brecha val−train | val final vs mínima | sobreajuste (regla) |
|---|---|---|---|---|---|---|---|
{tabla_val}

✎ **Interprete:** ¿en qué época dejó de mejorar la validación? ¿qué hace la brecha entre entrenamiento y validación al alargar el entrenamiento? ¿qué indica el hecho de que solo haya {len(val)} ejemplos de validación?

## 3. Calidad de las respuestas (base vs. adaptadores)
{tabla_fmt}

✎ **Interprete:** ¿qué cambió respecto a la línea base (cierre `Fuente:`/`Derivación:`, longitud, formato)? ¿hay respuestas con datos incorrectos o inventados? (cite 1–2 ejemplos de `comparacion.csv`). ¿Cómo respondieron los casos difíciles (fuera de alcance, ambiguos, mal escritos)?

## 4. Conclusión
✎ **Configuración recomendada y justificación:** …
✎ **Limitaciones** (tamaño del dataset, validación de {len(val)} ejemplos, una sola semilla): …
✎ **Mejoras propuestas:** …
"""
open("resultados/analisis_borrador.md", "w", encoding="utf-8").write(borrador)
print(borrador)

## 12. Exportar (paso 7)
**12.1 Adaptadores** (obligatorio): carpetas `adaptadores/adaptador_A` y `adaptador_B` (`adapter_model.safetensors` + `adapter_config.json`). Se empaquetan junto con los resultados.

In [ ]:
!rm -f entregables_p4.zip && zip -rq entregables_p4.zip adaptadores resultados
print("Contenido de entregables_p4.zip:")
!unzip -l entregables_p4.zip | tail -n +4 | head -30
from google.colab import files
files.download("entregables_p4.zip")
# Con Google Drive:  !cp entregables_p4.zip /content/drive/MyDrive/

**12.2 (Opcional) Fusionar con el modelo base, convertir a GGUF y cargar en Ollama.**
Cambie `EXPORTAR_GGUF = True` para ejecutarlo. Fusiona el adaptador elegido en el modelo **sin cuantizar** (fp16) y lo convierte a GGUF Q8_0. Es la ruta recomendada hacia Ollama: la documentación de Ollama advierte que un adaptador QLoRA aplicado sobre una base cuantizada distinta puede dar resultados erráticos.

In [ ]:
EXPORTAR_GGUF = False            # ← cámbielo a True para fusionar y convertir
MEJOR = min(DIAG, key=lambda n: DIAG[n]["val_min"])      # config con menor pérdida de validación (cámbielo si prefiere otra)
print("Adaptador a exportar:", MEJOR)

if EXPORTAR_GGUF:
    from peft import PeftModel
    base16 = AutoModelForCausalLM.from_pretrained(MODELO, device_map="cpu", **_kw_dtype(torch.float16))
    fusionado = PeftModel.from_pretrained(base16, f"adaptadores/adaptador_{MEJOR}").merge_and_unload()
    fusionado.save_pretrained("modelo_fusionado", safe_serialization=True)
    tokenizer.save_pretrained("modelo_fusionado")
    del base16, fusionado; gc.collect()
    print("Modelo fusionado guardado en modelo_fusionado/")

In [ ]:
if EXPORTAR_GGUF:
    !git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
    !pip install -q gguf sentencepiece
    !python llama.cpp/convert_hf_to_gguf.py modelo_fusionado --outfile modelo-rrhh-q8_0.gguf --outtype q8_0
    !ls -lh modelo-rrhh-q8_0.gguf

In [ ]:
if EXPORTAR_GGUF:
    # La plantilla de Ollama se genera a partir de la plantilla EXACTA usada en el entrenamiento (sin el BOS, que Ollama añade solo)
    plantilla = tokenizer.apply_chat_template(
        [{"role": "system", "content": "{{ .System }}"}, {"role": "user", "content": "{{ .Prompt }}"}],
        tokenize=False, add_generation_prompt=True)
    if tokenizer.bos_token and plantilla.startswith(tokenizer.bos_token):
        plantilla = plantilla[len(tokenizer.bos_token):]
    modelfile = ("FROM ./modelo-rrhh-q8_0.gguf\n"
                 'TEMPLATE """' + plantilla + '"""\n'
                 'SYSTEM """' + SISTEMA + '"""\n'
                 "PARAMETER temperature 0.2\nPARAMETER num_ctx 4096\n"
                 'PARAMETER stop "<|eot_id|>"\nPARAMETER stop "<|start_header_id|>"\n')
    open("Modelfile", "w", encoding="utf-8").write(modelfile)
    print(modelfile)
    !zip -q modelo_ollama.zip modelo-rrhh-q8_0.gguf Modelfile
    files.download("modelo_ollama.zip")

**Uso local en Ollama** (en su equipo, con `modelo-rrhh-q8_0.gguf` y `Modelfile` en la misma carpeta):
```powershell
ollama create rrhh-faq -f Modelfile
ollama run rrhh-faq "¿Con cuántos días de anticipación debo pedir mis vacaciones?"
```
Compare la respuesta con la que obtuvo en este cuaderno para el mismo prompt. **Alternativa más rápida (con salvedad):** `FROM llama3.2:3b` + `ADAPTER ./adaptadores/adaptador_A` en el `Modelfile`; Ollama la admite para Llama 3.2, pero un adaptador QLoRA sobre una base cuantizada distinta puede dar resultados irregulares, así que verifique las respuestas.

## 13. Lista de verificación de la entrega
| Entregable | Archivo |
|---|---|
| Cuaderno ejecutado y reproducible | este `.ipynb` guardado **con salidas** (`Archivo → Descargar → .ipynb`) |
| Adaptador guardado | `adaptadores/adaptador_A/`, `adaptadores/adaptador_B/` |
| Gráfico de pérdidas | `resultados/curvas_perdida.png` |
| Análisis de una página | `resultados/analisis_borrador.md` completado (✎) |
| Registro de métricas | `resultados/log_A.json`, `log_B.json`, `comparacion.csv`, `entorno.txt` |